In [0]:
%pip install duckdb

In [0]:
dbutils.library.restartPython()

In [0]:
import json
import urllib.request

import duckdb

# ESPG:4326 WGS 84
BBOX = {
    'xmin': 30.15,
    'xmax': 30.60,
    'ymin': 59.80,
    'ymax': 60.10
}

con = duckdb.connect()

con.execute("""
INSTALL httpfs;
LOAD httpfs;
""")

con.execute(f"""
SET s3_region='us-west-2';
""")

latest = con.execute(
    "SELECT latest FROM 'https://stac.overturemaps.org/catalog.json'"
).fetchone()[0]
base = f"s3://overturemaps-us-west-2/release/{latest}"
print("release:", latest)

where = f"""
    bbox.xmin BETWEEN {BBOX['xmin']} AND {BBOX['xmax']}
    AND bbox.ymin BETWEEN {BBOX['ymin']} AND {BBOX['ymax']}"""

num_places = con.execute(f"""
SELECT COUNT(*) AS n
FROM read_parquet('{base}/theme=places/type=place/*', hive_partitioning=1)
WHERE {where}
""").fetchone()[0]
print(f"places: {num_places}")

num_buildings = con.execute(f"""
SELECT COUNT(*) AS n
FROM read_parquet('{base}/theme=buildings/type=building/*', hive_partitioning=1)
WHERE {where}
""").fetchone()[0]
print(f"buildings: {num_buildings}")








In [0]:
rows = con.execute(f"""
WITH b AS (
    SELECT floor(confidence * 5) / 5 AS bucket, COUNT(*) AS places
    FROM read_parquet('{base}/theme=places/type=place/*', hive_partitioning=1)
    WHERE {where}
    GROUP BY bucket
)
SELECT bucket, places,
       SUM(places) OVER (ORDER BY bucket DESC NULLS LAST) AS at_or_above
FROM b
ORDER BY bucket NULLS FIRST
""").fetchall()
for bucket, places, at_or_above in rows:
    label = "no score" if bucket is None else f"{bucket}"
    print(f"{label}: {places}  (cumulative from top: {at_or_above})")

In [0]:
p = spark.read.parquet(f"{base}/theme=places/type=place/")
p.printSchema()

In [0]:
import json, urllib.request
from pyspark.sql import functions as F

latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"

places = (
    spark.read.parquet(f"{base}/theme=places/type=place/")
    .where("bbox.xmin BETWEEN 30.15 AND 30.60 AND bbox.ymin BETWEEN 59.80 AND 60.10")
    .select(
        "id",
        "taxonomy",
        "bbox",
        F.col("names.primary").alias("name"),
        "basic_category",
        "confidence"
    )
)

buildings = (
    spark.read.parquet(f"{base}/theme=buildings/type=building/")
    .where("bbox.xmin BETWEEN 30.15 AND 30.60 AND bbox.ymin BETWEEN 59.80 AND 60.10")
    .select(
        "id",
        "geometry",
        "bbox",
        F.col("names.primary").alias("name"),
        "subtype",
        F.col("class").alias("building_class"),
    )
)

buildings.count() 
display(buildings.limit(10))
places.count()   
display(places)

In [0]:
b = spark.read.parquet(f"{base}/theme=buildings/type=building/")
b.printSchema()

In [0]:
import json, urllib.request
from pyspark.sql import functions as F



latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"

buildings = (
    spark.read.parquet(f"{base}/theme=buildings/type=building/")
    .where("bbox.xmin BETWEEN 30.15 AND 30.60 AND bbox.ymin BETWEEN 59.80 AND 60.10")
    .select(
        "id",
        "geometry",
        "bbox",
        F.col("names.primary").alias("name"),
        "subtype",
        F.col("class").alias("building_class"),
    )
)
buildings.count() 
display(buildings.limit(10))

In [0]:
%sql
SELECT current_catalog(), current_schema()

In [0]:
import json 
import urllib.request
from pyspark.sql import functions as F


spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.spb")
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.spb.landing")

LANDING_DIR = "/Volumes/workspace/spb/landing"

BBOX = {
    'xmin': 30.15,
    'xmax': 30.60,
    'ymin': 59.80,
    'ymax': 60.10
}

where = f"""
    bbox.xmin BETWEEN {BBOX['xmin']} AND {BBOX['xmax']}
    AND bbox.ymin BETWEEN {BBOX['ymin']} AND {BBOX['ymax']}
    """


latest = json.load(urllib.request.urlopen("https://stac.overturemaps.org/catalog.json"))["latest"]
base = f"s3://overturemaps-us-west-2/release/{latest}"


places = (
    spark.read.parquet(f"{base}/theme=places/type=place/")
    .where(where)
    .select(
        "id",
        "taxonomy",
        "bbox",
        F.col("names.primary").alias("name"),
        "basic_category",
        "confidence")
        .write.mode("overwrite").parquet(f"{LANDING_DIR}/places")
)


buildings = (
    spark.read.parquet(f"{base}/theme=buildings/type=building/")
    .where(where)
    .select(
        "id",
        "geometry",
        "bbox",
        F.col("names.primary").alias("name"),
        "subtype",
        F.col("class").alias("building_class"),
    )
    .write.mode("overwrite").parquet(f"{LANDING_DIR}/buildings")
)

In [0]:
%sql
SELECT * FROM read_files("/Volumes/workspace/spb/landing/places")